In [1]:
#| default_exp data.dataset

# Data

A text-to-image ReID experiment needs three things:

- **train**: `(pid, image_path, caption)` triples, one per caption, with pids relabelled to `0..num_train_pids-1` so they can index an ID-classification head;
- **query**: `(pid, caption)`, one per test caption: the text queries;
- **gallery**: `(pid, image_path)`, one per test image: what every query is ranked against. A query's correct matches are the gallery images with the same pid.

`T2IReIDDataset` builds those lists from an annotation file in the standard T2I-ReID format (a JSON list of records with `split`, `captions`, `file_path` and `id`, as in CUHK-PEDES), and `build_dataloaders` wraps them in PyTorch loaders.

In [ ]:
#| export
import json
from pathlib import Path
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import DataLoader, Dataset

## Parent dataset

In [ ]:
#| export
class T2IReIDDataset:
    
    """Splits a T2I-ReID annotation file into train / query / gallery lists.

    Subclasses only need to point it at a dataset (`root`, `ann_file`). Fields other than split, captions,
    file_path and id (masks, part boxes, ...) are ignored here.
    """

    def __init__(self, root, ann_file, test_split='val'):
        self.root = Path(root)
        records = json.loads((self.root / ann_file).read_text())
        train = [r for r in records if r['split'] == 'train']
        test = [r for r in records if r['split'] == test_split]

        pid2label = {pid: label for label, pid in enumerate(sorted({r['id'] for r in train}))}
        self.train = [(pid2label[r['id']], str(self.root / r['file_path']), caption)
                      for r in train for caption in r['captions']]
        self.query = [(r['id'], caption) for r in test for caption in r['captions']]
        self.gallery = [(r['id'], str(self.root / r['file_path'])) for r in test]
        self.num_train_pids = len(pid2label)

    def __repr__(self):
        n_ids = lambda samples: len({s[0] for s in samples})
        return (f'{type(self).__name__}\n'
                f'  train:   {len(self.train):6} captions | {n_ids(self.train):5} ids\n'
                f'  query:   {len(self.query):6} captions | {n_ids(self.query):5} ids\n'
                f'  gallery: {len(self.gallery):6} images   | {n_ids(self.gallery):5} ids')


class T2IVeRW(T2IReIDDataset):
    """T2I-VeRW: vehicle images with one caption each; train/val split by vehicle id (70/30)."""

    def __init__(self, root='/mnt/data/ishant/datasets/T2I-VeRW',
                 ann_file='annotations/reid_with_mask_prompt_and_boxes_filepath_prefixid_idsplit_70_30.json',
                 test_split='val'):
        super().__init__(root, ann_file, test_split)

In [4]:
dataset = T2IVeRW()
dataset

T2IVeRW
  train:    10264 captions |  1797 ids
  query:     4404 captions |   769 ids
  gallery:   4404 images   |   769 ids

In [5]:
dataset.train[0], dataset.query[0], dataset.gallery[0]

((0,
  '/mnt/data/ishant/datasets/T2I-VeRW/images/00004_000016.jpg',
  'Dark gray station wagon with black rear window, horizontal red tail lights on each side, silver trim strip along lower rear bumper, and multi-spoke silver wheels; small black fin-shaped object centered on rear roof edge; rear door features black badge with white lettering on left side and smaller badge on right side; side windows have silver trim along top edge.'),
 (1,
  'White SUV with a silver chrome grille containing horizontal bars and a central emblem, willow-leaf shaped headlights, five-spoke silver wheels, black roof rails along the roof edges, and a black sunroof panel; two blue illuminated daytime running lights are present on the lower front bumper, and a white tissue box is visible on the dashboard through the front windshield.'),
 (1, '/mnt/data/ishant/datasets/T2I-VeRW/images/00001_000001.jpg'))

## PyTorch datasets

In [6]:
#| export
class ImageTextDataset(Dataset):
    """Training pairs: {'pid', 'image', 'caption'}."""

    def __init__(self, samples, transform=None, tokenizer=None):
        self.samples, self.transform, self.tokenizer = samples, transform, tokenizer

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        pid, image_path, caption = self.samples[i]
        return {'pid': pid, 'image': _load_image(image_path, self.transform), 'caption': _tokenize(caption, self.tokenizer)}


class ImageDataset(Dataset):
    """Gallery images: {'pid', 'image'}."""

    def __init__(self, samples, transform=None):
        self.samples, self.transform = samples, transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        pid, image_path = self.samples[i]
        return {'pid': pid, 'image': _load_image(image_path, self.transform)}


class TextDataset(Dataset):
    """Text queries: {'pid', 'caption'}."""

    def __init__(self, samples, tokenizer=None):
        self.samples, self.tokenizer = samples, tokenizer

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        pid, caption = self.samples[i]
        return {'pid': pid, 'caption': _tokenize(caption, self.tokenizer)}


def _load_image(path, transform):
    image = Image.open(path).convert('RGB')
    return transform(image) if transform else image


def _tokenize(caption, tokenizer):
    # tokenizer: str -> 1-D LongTensor. Without one, the raw string is returned (and batched as a list).
    return tokenizer(caption) if tokenizer else caption

## Transforms and loaders

Images are resized straight to `size` (no crop, so the whole vehicle stays in view) and normalised with CLIP's statistics. There's deliberately no horizontal flip: captions describe sides ("sticker in the upper left of the windshield"), and flipping the image would contradict them.

In [7]:
#| export
CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD = (0.26862954, 0.26130258, 0.27577711)


def build_transform(size=(224, 224), mean=CLIP_MEAN, std=CLIP_STD):
    return T.Compose([T.Resize(size, interpolation=T.InterpolationMode.BICUBIC), T.ToTensor(), T.Normalize(mean, std)])


def build_dataloaders(dataset, tokenizer=None, transform=None, batch_size=64, test_batch_size=256, num_workers=8):
    """Returns (train_loader, query_loader, gallery_loader) for a T2IReIDDataset."""
    transform = transform or build_transform()
    loader = lambda ds, bs, shuffle: DataLoader(ds, batch_size=bs, shuffle=shuffle, drop_last=shuffle,
                                                num_workers=num_workers, pin_memory=True)
    return (loader(ImageTextDataset(dataset.train, transform, tokenizer), batch_size, True),
            loader(TextDataset(dataset.query, tokenizer), test_batch_size, False),
            loader(ImageDataset(dataset.gallery, transform), test_batch_size, False))

With CLIP, use its tokenizer. Captions here run up to ~130 words, past CLIP's 77-token context, so `truncate=True` is needed (it cuts the caption and keeps the end-of-text token).

In [8]:
import clip

tokenizer = lambda caption: clip.tokenize(caption, truncate=True)[0]
train_loader, query_loader, gallery_loader = build_dataloaders(dataset, tokenizer)

batch = next(iter(train_loader))
{k: (v.shape, v.dtype) for k, v in batch.items()}

{'pid': (torch.Size([64]), torch.int64),
 'image': (torch.Size([64, 3, 224, 224]), torch.float32),
 'caption': (torch.Size([64, 77]), torch.int32)}

In [9]:
q, g = next(iter(query_loader)), next(iter(gallery_loader))
print('query  :', q['caption'].shape, q['pid'][:8].tolist())
print('gallery:', g['image'].shape, g['pid'][:8].tolist())
print(len(train_loader), 'train batches |', len(query_loader), 'query batches |', len(gallery_loader), 'gallery batches')

query  : torch.Size([256, 77]) [1, 1, 1, 1, 1, 1, 17, 17]
gallery: torch.Size([256, 3, 224, 224]) [1, 1, 1, 1, 1, 1, 17, 17]
160 train batches | 18 query batches | 18 gallery batches
